In [0]:
BASE = "workspace.retail"

# 1. Combine the Silver tables into a reusable sales view.
# Each row represents one item line from a qualifying order.
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW sales_lines AS
    SELECT
        o.order_id,
        oi.order_item_id,
        CAST(o.created_at AS DATE) AS order_date,
        c.customer_id,
        c.customer_name,
        s.store_id,
        s.store_city,
        p.product_id,
        p.product_name,
        cat.category_id,
        cat.category_name,
        oi.quantity,
        oi.unit_price,
        oi.quantity * oi.unit_price AS line_sales
    FROM {BASE}.silver_orders o
    JOIN {BASE}.silver_order_items oi
        ON o.order_id = oi.order_id
    JOIN {BASE}.silver_customers c
        ON o.customer_id = c.customer_id
    JOIN {BASE}.silver_stores s
        ON o.store_id = s.store_id
    JOIN {BASE}.silver_products p
        ON oi.product_id = p.product_id
    JOIN {BASE}.silver_categories cat
        ON p.category_id = cat.category_id
    WHERE o.status IN ('PAID', 'SHIPPED')
""")

# 2. Define the report queries.
reports = {
    "gold_daily_sales": """
        SELECT
            order_date,
            COUNT(DISTINCT order_id) AS total_orders,
            SUM(quantity) AS units_sold,
            SUM(line_sales) AS total_sales,
            ROUND(
                SUM(line_sales) / COUNT(DISTINCT order_id), 2
            ) AS average_order_value
        FROM sales_lines
        GROUP BY order_date
    """,

    "gold_store_performance": """
        SELECT
            store_id,
            store_city,
            COUNT(DISTINCT order_id) AS total_orders,
            SUM(quantity) AS units_sold,
            SUM(line_sales) AS total_sales
        FROM sales_lines
        GROUP BY store_id, store_city
    """,

    "gold_product_performance": """
        SELECT
            product_id,
            product_name,
            COUNT(DISTINCT order_id) AS orders_containing_product,
            SUM(quantity) AS units_sold,
            SUM(line_sales) AS total_sales
        FROM sales_lines
        GROUP BY product_id, product_name
    """,

    "gold_customer_metrics": """
        SELECT
            customer_id,
            customer_name,
            COUNT(DISTINCT order_id) AS total_orders,
            SUM(line_sales) AS total_spent,
            MIN(order_date) AS first_order_date,
            MAX(order_date) AS latest_order_date
        FROM sales_lines
        GROUP BY customer_id, customer_name
    """,

    "gold_category_revenue": """
        SELECT
            category_id,
            category_name,
            SUM(quantity) AS units_sold,
            SUM(line_sales) AS total_sales
        FROM sales_lines
        GROUP BY category_id, category_name
    """,
}

# 3. Save the reports as Delta tables.
for table_name, query in reports.items():
    (
        spark.sql(query)
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{BASE}.{table_name}")
    )

    print(f"Saved: {BASE}.{table_name}")

# 4. Check that daily and store summaries match the sales detail.
totals = spark.sql(f"""
    SELECT
        (SELECT COALESCE(SUM(line_sales), 0)
         FROM sales_lines) AS detail_total,

        (SELECT COALESCE(SUM(total_sales), 0)
         FROM {BASE}.gold_daily_sales) AS daily_total,

        (SELECT COALESCE(SUM(total_sales), 0)
         FROM {BASE}.gold_store_performance) AS store_total
""").first()

assert (
    totals["detail_total"]
    == totals["daily_total"]
    == totals["store_total"]
), "Sales totals do not match."

print("Passed: detail, daily, and store sales totals match.")

# 5. Display your daily sales report.
display(spark.sql(f"""
    SELECT *
    FROM {BASE}.gold_daily_sales
    ORDER BY order_date DESC
"""))

Saved: workspace.retail.gold_daily_sales
Saved: workspace.retail.gold_store_performance
Saved: workspace.retail.gold_product_performance
Saved: workspace.retail.gold_customer_metrics
Saved: workspace.retail.gold_category_revenue
Passed: detail, daily, and store sales totals match.


order_date,total_orders,units_sold,total_sales,average_order_value
2026-10-17,336,659,36245000.00,107872.02
2026-10-16,6,23,1120000.00,186666.67
2026-10-15,9,34,1770000.00,196666.67
2026-10-14,6,27,1425000.00,237500.00
2026-10-13,6,28,1515000.00,252500.00
2026-10-12,4,18,915000.00,228750.00
2026-10-11,13,41,2265000.00,174230.77
2026-10-10,9,38,2085000.00,231666.67
2026-10-09,6,28,1580000.00,263333.33
2026-10-08,5,19,935000.00,187000.00
